# 05 - 异常处理与文件操作（Java vs Python）

Java 开发者对 try/catch 很熟，Python 差异不大但有几个关键不同。

## 今日目标（15-20 分钟）

- 掌握 `try/except/else/finally` 的写法差异
- 了解 Python 常见异常类型
- 会用 `with` 语句管理资源（对比 Java 的 try-with-resources）
- 会读写文件

完成标准：通过末尾打卡题

## 1. try / except / else / finally

In [ ]:
# 说明：演示异常处理流程，区分正常路径与错误路径，提升代码健壮性。

# Java:
# try {
#     int x = 10 / 0;
# } catch (ArithmeticException e) {
#     System.out.println(e.getMessage());
# } finally {
#     System.out.println("done");
# }

# Python：catch → except，其他几乎一样
try:
    x = 10 / 0
except ZeroDivisionError as e:
    print(f"caught: {e}")
finally:
    print("done")


In [ ]:
# 说明：演示异常处理流程，区分正常路径与错误路径，提升代码健壮性。

# else 块：只有 try 没有异常时才执行（Java 没有这个）
# 好处：把"正常逻辑"和"异常处理"分开，代码更清晰

def safe_divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        print("cannot divide by zero")
    else:
        # 只有没异常时才走这里
        print(f"result: {result}")
    finally:
        # 无论如何都走这里
        print("---")

safe_divide(10, 3)
safe_divide(10, 0)

# 💡 实战：else 块的核心价值是"避免误捕获"——
# 如果把 print(f"result: {result}") 放进 try 块里，
# 万一 print 本身抛出异常，也会被 ZeroDivisionError 的 except 捕获（虽然极少见）
# 用 else 能确保：except 里只捕获 try 里明确预期的异常，正常逻辑的错误该暴露就暴露


In [ ]:
# 说明：演示异常处理流程，区分正常路径与错误路径，提升代码健壮性。

# 捕获多种异常

# 方式一：分别处理
try:
    nums = [1, 2, 3]
    print(nums[10])
except IndexError as e:
    print(f"IndexError: {e}")
except KeyError as e:
    print(f"KeyError: {e}")

# 方式二：合并处理（Java 7+ 也支持 catch(A | B e)）
try:
    nums = [1, 2, 3]
    print(nums[10])
except (IndexError, KeyError) as e:
    print(f"caught: {e}")


In [ ]:
# 说明：演示异常处理流程，区分正常路径与错误路径，提升代码健壮性。

# 捕获所有异常：用 Exception（对比 Java 的 catch(Exception e)）
# 不推荐滥用，会吞掉真正的 bug

try:
    x = int("abc")
except Exception as e:
    print(f"{type(e).__name__}: {e}")  # ValueError: invalid literal...

# ⚠️ 坑：except Exception 并不能捕获"所有"异常——
# Python 异常层级：BaseException → Exception → 各种具体异常
# KeyboardInterrupt（Ctrl+C）和 SystemExit 继承自 BaseException，不是 Exception
# 所以 except Exception 不会捕获 Ctrl+C，这是正确行为（用户能正常中断程序）
#
# 但如果你写 except BaseException: 或裸 except:，就会吞掉 Ctrl+C，
# 导致程序无法被手动中断——这是面试高频题，也是生产环境的严重 bug


## 2. 主动抛出异常 + 自定义异常

In [ ]:
# 说明：演示异常处理流程，区分正常路径与错误路径，提升代码健壮性。

# Java: throw new IllegalArgumentException("...");
# Python: raise ValueError("...")

def set_age(age: int):
    if age < 0:
        raise ValueError(f"age cannot be negative: {age}")
    return age

try:
    set_age(-1)
except ValueError as e:
    print(e)


In [ ]:
# 说明：演示自定义异常在业务校验中的用法，便于精确区分错误类型。

# 自定义异常：继承 Exception 就行（对比 Java extends RuntimeException）

class NotFoundError(Exception):
    pass  # 最简单的自定义异常，只要一个 pass

class BusinessError(Exception):
    def __init__(self, code: int, message: str):
        super().__init__(message)
        self.code = code

try:
    raise BusinessError(404, "user not found")
except BusinessError as e:
    print(f"code={e.code}, msg={e}")


## 3. 常见异常类型

| Python 异常 | Java 类比 | 触发场景 |
|------------|-----------|----------|
| `ValueError` | `IllegalArgumentException` | 参数值不合法 |
| `TypeError` | `ClassCastException` | 类型不匹配 |
| `KeyError` | `NoSuchElementException` | dict 中 key 不存在 |
| `IndexError` | `ArrayIndexOutOfBoundsException` | 列表索引越界 |
| `AttributeError` | `NoSuchFieldException` | 访问不存在的属性/方法 |
| `FileNotFoundError` | `FileNotFoundException` | 文件不存在 |
| `IOError` | `IOException` | IO 操作失败 |
| `RuntimeError` | `RuntimeException` | 通用运行时错误 |
| `StopIteration` | `NoSuchElementException` | 迭代器耗尽 |

Python 没有 checked exception（Java 的 throws 声明），所有异常都是 unchecked。

## 4. with 语句（上下文管理器）

In [ ]:
# 说明：演示 with 自动管理文件资源，避免忘记 close 导致资源泄露。

# Java 7+: try-with-resources
# try (BufferedReader br = new BufferedReader(new FileReader("test.txt"))) {
#     String line = br.readLine();
# }

# Python: with 语句，自动关闭资源，不需要手动 close()

# 先创建一个测试文件
with open("test.txt", "w") as f:
    f.write("line 1\n")
    f.write("line 2\n")
    f.write("line 3\n")
# 出了 with 块，f 自动关闭

print("file written")


In [ ]:
# 说明：演示 with 自动管理文件资源，避免忘记 close 导致资源泄露。

# 读文件

# 方式一：一次读全部
with open("test.txt", "r") as f:
    content = f.read()
print(content)

# 方式二：按行读（大文件推荐，不会一次性加载到内存）
with open("test.txt", "r") as f:
    for line in f:
        print(line.strip())  # strip() 去掉末尾换行符


In [ ]:
# 说明：演示 with 自动管理文件资源，避免忘记 close 导致资源泄露。

# 读全部行到 list
with open("test.txt", "r") as f:
    lines = f.readlines()  # 返回 list[str]

print(lines)         # ['line 1\n', 'line 2\n', 'line 3\n']
print(len(lines))    # 3


In [ ]:
# 说明：演示 with 自动管理文件资源，避免忘记 close 导致资源泄露。

# 追加写入（"a" 模式，对比 "w" 会覆盖）
with open("test.txt", "a") as f:
    f.write("line 4\n")

with open("test.txt", "r") as f:
    print(f.read())

# ⚠️ 坑："w" 模式（write）会在 open() 的瞬间清空文件，不需要等 f.write()！
# 只要执行 open("file.txt", "w")，原内容就已经没了——即使你马上关闭文件也无法恢复
# 常见模式对比：
# "r"  → 只读（默认，文件不存在会报错）
# "w"  → 覆盖写（危险：立即清空已有内容）
# "a"  → 追加写（安全：不清空，写在末尾）
# "r+" → 读写（不清空，从头开始写）


## 5. JSON 文件读写（AI 开发高频）

In [ ]:
# 说明：演示 JSON 文件读取并转为 dict，这是接口开发高频操作。

import json

# 写 JSON
data = {"name": "kai", "skills": ["python", "java"], "score": 95}

with open("user.json", "w") as f:
    json.dump(data, f, indent=2)  # indent=2 格式化输出，方便阅读

# 读 JSON
with open("user.json", "r") as f:
    loaded = json.load(f)  # 直接得到 dict

print(loaded)
print(type(loaded))  # <class 'dict'>
print(loaded["skills"])


In [ ]:
# 说明：演示 JSON 与 dict 的读写/互转，这是接口开发中的高频操作。

# JSON 字符串互转（不涉及文件，处理 API 响应时常用）
# Java: new ObjectMapper().writeValueAsString(obj) / readValue(str, Map.class)

import json

# dict → JSON 字符串
json_str = json.dumps({"name": "kai", "age": 18})
print(json_str)        # {"name": "kai", "age": 18}
print(type(json_str))  # <class 'str'>

# JSON 字符串 → dict
obj = json.loads(json_str)
print(obj)             # {'name': 'kai', 'age': 18}
print(type(obj))       # <class 'dict'>

# 注意命名规律：
# dump/load   → 文件（File）
# dumps/loads → 字符串（String，s 代表 string）

# 💡 实战：调用 OpenAI / Claude 等 AI API 时，全链路 JSON 处理模式：
#
# import httpx
# response = httpx.post(url, json=payload)  # httpx 自动 json.dumps 请求体
# data = response.json()                    # 自动 json.loads 响应体 → dict
#
# 如果拿到的是裸字节流（如 WebSocket / 流式响应）：
# raw = b'{"choices": [{"message": {"content": "hello"}}]}'
# data = json.loads(raw.decode("utf-8"))    # bytes → str → dict
# text = data["choices"][0]["message"]["content"]


## 6. pathlib（现代文件路径操作）

In [ ]:
# 说明：演示 pathlib 路径操作，通常比 os.path 更直观、可读。

# Java: Paths.get("/home/kai").resolve("projects")
# Python: 老写法用 os.path，现代写法用 pathlib

from pathlib import Path

p = Path(".")  # 当前目录
print(p.resolve())           # 绝对路径

# 拼接路径：用 / 运算符（很 Pythonic）
config_path = Path("config") / "settings.json"
print(config_path)           # config/settings.json

# 常用操作
f = Path("test.txt")
print(f.exists())            # True
print(f.name)                # test.txt
print(f.stem)                # test（不带扩展名）
print(f.suffix)              # .txt
print(f.parent)              # .（父目录）


In [ ]:
# 说明：演示 pathlib 路径操作，通常比 os.path 更直观、可读。

# pathlib 也能直接读写文件（小文件方便）
from pathlib import Path

p = Path("quick.txt")
p.write_text("hello pathlib")  # 写
content = p.read_text()        # 读
print(content)

# 清理测试文件
Path("test.txt").unlink(missing_ok=True)
Path("user.json").unlink(missing_ok=True)
Path("quick.txt").unlink(missing_ok=True)
print("cleaned up")


## 7. Java 对照速记

| Java | Python |
|------|--------|
| `catch` | `except` |
| `throw` | `raise` |
| `throws`（checked exception） | 不存在，所有异常都是 unchecked |
| `try-with-resources` | `with` 语句 |
| `ObjectMapper` | `json.dump/load/dumps/loads` |
| `Paths.get().resolve()` | `Path() / "subdir"` |
| 无 | `try...else`（没异常时执行） |

## 今日打卡题

1. 写函数 `safe_parse_int(s: str) -> int | None`，尝试把字符串转 int，失败返回 None（不要让异常抛出去）
2. 用 `with` + `json` 模块，把 `{"items": [1, 2, 3]}` 写入 `data.json`，再读出来打印
3. 自定义一个 `ApiError(code, message)` 异常，手动 raise 并 catch 它，打印 code 和 message

In [ ]:
# 说明：这是练习留空区域，建议先独立完成，再运行后续参考答案进行对照。

# TODO: 先自己写


In [ ]:
# 说明：这是打卡题参考实现，建议先自己做一遍，再用它核对思路和语法。

# 参考答案（先自己写完再看）
import json
from pathlib import Path

# 1
def safe_parse_int(s: str) -> int | None:
    try:
        return int(s)
    except ValueError:
        return None

print(safe_parse_int("42"))     # 42
print(safe_parse_int("abc"))    # None

# 2
with open("data.json", "w") as f:
    json.dump({"items": [1, 2, 3]}, f)

with open("data.json", "r") as f:
    data = json.load(f)
print(data)  # {'items': [1, 2, 3]}

Path("data.json").unlink(missing_ok=True)  # 清理

# 3
class ApiError(Exception):
    def __init__(self, code: int, message: str):
        super().__init__(message)
        self.code = code

try:
    raise ApiError(500, "internal server error")
except ApiError as e:
    print(f"code={e.code}, msg={e}")


下一步建议：进入 `06-module-package.ipynb`，掌握 import 机制和项目结构。